In [ ]:
# -*- coding: utf-8 -*-
"""
Answer Domain Classification Script
This script classifies Hindi QA answers into different knowledge domains
"""

import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from collections import Counter
from tqdm import tqdm

# Load the CSV file with predictions
print("Loading prediction data...")
df = pd.read_csv("predictions(pre-trained+finetuned).csv", encoding="utf-8-sig")

# Display sample data
print(f"Loaded {len(df)} predictions")
print("\nSample data:")
print(df[["Question", "Predicted_Answer"]].head(3))



Loading prediction data...
Loaded 500 predictions

Sample data:
                                     Question  \
0  पहले सफल ऑटोमोबाइल का आविष्कार किसने किया?   
1            डीएनए की संरचना की खोज किसने की?   
2             टेलीफोन का आविष्कार किसने किया?   

                                    Predicted_Answer  
0  किंग लेविस ने पहले सफल ऑटोमोबाइल का आविष्कार क...  
1           जेनेटिक पारमा डीएनए की संरचना की खोज की।  
2  अलेक्संडर ग्रेहम बेल टेलीफोन के आविष्कार के लि...  


In [ ]:
# -*- coding: utf-8 -*-
"""
Answer Domain Classification Script (Fixed)
This script classifies Hindi QA answers into different knowledge domains
"""

import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from collections import Counter
from tqdm import tqdm

# Load the CSV file with predictions
print("Loading prediction data...")
df = pd.read_csv("predictions(pre-trained+finetuned).csv", encoding="utf-8-sig")

# Display sample data
print(f"Loaded {len(df)} predictions")
print("\nSample data:")
print(df[["Question", "Predicted_Answer"]].head(3))

# Define domain keywords (Hindi & English)
domain_keywords = {
    "History": ["इतिहास", "राजा", "महाराजा", "युद्ध", "साम्राज्य", "राज्य", "प्राचीन", "स्वतंत्रता", "emperor", "king", "dynasty", "war", "ancient", "freedom", "independence", "revolt", "kingdom"],

    "Geography": ["भूगोल", "नदी", "पहाड़", "महासागर", "राज्य", "देश", "प्रदेश", "राजधानी", "mountain", "river", "ocean", "country", "state", "capital", "continent", "climate", "plateau", "geography"],

    "Politics": ["राजनीति", "प्रधानमंत्री", "मुख्यमंत्री", "राष्ट्रपति", "संसद", "लोकसभा", "राज्यसभा", "minister", "president", "parliament", "constitution", "government", "election", "party", "politics"],

    "Science": ["विज्ञान", "भौतिक", "रसायन", "जीव", "आविष्कार", "physics", "chemistry", "biology", "invention", "discovery", "scientist", "element", "compound", "science", "technology", "research"],

    "Sports": ["खेल", "क्रिकेट", "फुटबॉल", "हॉकी", "खिलाड़ी", "cricket", "football", "hockey", "player", "tournament", "championship", "medal", "olympics", "sports", "game", "athlete"],

    "Economics": ["अर्थशास्त्र", "अर्थव्यवस्था", "मुद्रा", "बैंक", "वित्त", "economy", "finance", "bank", "currency", "market", "trade", "economics", "budget", "gdp", "economic", "financial"],

    "Culture": ["संस्कृति", "त्योहार", "परंपरा", "भाषा", "साहित्य", "कला", "culture", "festival", "tradition", "language", "literature", "art", "custom", "heritage", "folk", "dance", "music"],

    "Religion": ["धर्म", "मंदिर", "मस्जिद", "चर्च", "पूजा", "religion", "temple", "mosque", "church", "worship", "god", "deity", "spiritual", "holy", "sacred", "prayer", "ritual"],

    "Technology": ["प्रौद्योगिकी", "कंप्यूटर", "इंटरनेट", "सॉफ्टवेयर", "हार्डवेयर", "computer", "internet", "software", "hardware", "technology", "digital", "electronic", "program", "device", "app"]
}

def classify_domain(text):
    """
    Classify a text into one of the predefined domains based on keyword presence
    Returns the domain with the highest keyword match count
    """
    # Convert to lowercase for easier matching
    if not isinstance(text, str):
        return "Unknown"

    text = text.lower()

    # Count matches for each domain
    domain_scores = {}
    for domain, keywords in domain_keywords.items():
        score = sum(1 for keyword in keywords if keyword.lower() in text.lower())
        domain_scores[domain] = score

    # Return the domain with the highest score
    if max(domain_scores.values()) > 0:
        return max(domain_scores.items(), key=lambda x: x[1])[0]
    else:
        return "Other"  # If no keywords match

# Apply classification to each predicted answer
print("\nClassifying answers by domain...")
tqdm.pandas(desc="Processing")
df["Domain"] = df["Predicted_Answer"].progress_apply(classify_domain)

# Count the number of answers in each domain
domain_counts = df["Domain"].value_counts()
print("\nDomain distribution:")
print(domain_counts)

# Save the classified data
df.to_csv("classified_predictions_llama3.csv", index=False, encoding="utf-8-sig")
print("\nClassified data saved to 'classified_predictions_llama3.csv'")

# Generate a bar chart of domain distribution
plt.figure(figsize=(12, 6))
domain_counts.plot(kind='bar', color='skyblue')
plt.title('Distribution of Answer Domains')
plt.xlabel('Domain')
plt.ylabel('Number of Answers')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('domain_distribution.png')
plt.close()

print("Domain distribution chart saved as 'domain_distribution.png'")

# Optional: Advanced clustering analysis
print("\nPerforming advanced clustering analysis...")

# Vectorize the text data
print("Vectorizing text data...")
# Only include answers that have at least 5 characters
valid_answers = df[df["Predicted_Answer"].str.len() > 5]["Predicted_Answer"].fillna("")
vectorizer = TfidfVectorizer(max_features=1000, stop_words=['और', 'है', 'हैं', 'का', 'के', 'की', 'में', 'से', 'पर', 'एक', 'यह', 'कि', 'थे', 'थी', 'थी', 'हो', 'था'])
X = vectorizer.fit_transform(valid_answers)

# Apply K-means clustering
print("Applying K-means clustering...")
n_clusters = min(10, len(valid_answers))  # Ensure we don't have more clusters than samples
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
df_subset = df[df["Predicted_Answer"].str.len() > 5].copy()
df_subset["Cluster"] = kmeans.fit_predict(X)

# Generate top words for each cluster
cluster_keywords = {}
feature_names = vectorizer.get_feature_names_out()

print("\nTop keywords for each cluster:")
for i in range(n_clusters):
    cluster_docs = df_subset[df_subset["Cluster"] == i]["Predicted_Answer"]
    if len(cluster_docs) > 0:
        cluster_tfidf = vectorizer.transform(cluster_docs)
        # Fix: Use numpy's mean function on the sparse matrix after converting to array
        tfidf_mean = np.mean(cluster_tfidf.toarray(), axis=0)

        # Get indices of top 10 words
        top_indices = np.argsort(tfidf_mean)[-10:]

        # Get the words corresponding to these indices
        top_words = [feature_names[idx] for idx in top_indices]

        cluster_keywords[i] = top_words
        print(f"Cluster {i} ({len(cluster_docs)} answers): {', '.join(top_words)}")

# Generate a scatter plot of the clusters
print("\nGenerating cluster visualization...")
pca = PCA(n_components=2)
X_dense = X.toarray()
pca_result = pca.fit_transform(X_dense)

plt.figure(figsize=(12, 8))
plt.scatter(pca_result[:, 0], pca_result[:, 1], c=df_subset["Cluster"], cmap='viridis', alpha=0.5)
plt.title('PCA Cluster Visualization of Answers')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.colorbar(label='Cluster')
plt.savefig('answer_clusters.png')
plt.close()

print("Cluster visualization saved as 'answer_clusters.png'")

# Optional: Compare manual classification with clustering
print("\nComparing manual classification with clustering...")
cluster_domain_mapping = {}
for cluster in range(n_clusters):
    cluster_domains = df_subset[df_subset["Cluster"] == cluster]["Domain"].value_counts()
    if not cluster_domains.empty:
        cluster_domain_mapping[cluster] = cluster_domains.index[0]
        print(f"Cluster {cluster} most common domain: {cluster_domains.index[0]} ({cluster_domains.iloc[0]} answers)")

# Create a cross-tabulation
cross_tab = pd.crosstab(df_subset["Cluster"], df_subset["Domain"])
print("\nCluster vs Domain Cross-tabulation:")
print(cross_tab)

print("\nAnalysis complete!")

Loading prediction data...
Loaded 500 predictions

Sample data:
                                     Question  \
0  पहले सफल ऑटोमोबाइल का आविष्कार किसने किया?   
1            डीएनए की संरचना की खोज किसने की?   
2             टेलीफोन का आविष्कार किसने किया?   

                                    Predicted_Answer  
0  किंग लेविस ने पहले सफल ऑटोमोबाइल का आविष्कार क...  
1           जेनेटिक पारमा डीएनए की संरचना की खोज की।  
2  अलेक्संडर ग्रेहम बेल टेलीफोन के आविष्कार के लि...  

Classifying answers by domain...


Processing: 100%|██████████| 500/500 [00:00<00:00, 2966.94it/s]


Domain distribution:
Domain
Other         338
Science       134
Religion       14
Technology      8
Unknown         2
Geography       2
Culture         1
History         1
Name: count, dtype: int64



Classified data saved to 'classified_predictions_llama3.csv'
Domain distribution chart saved as 'domain_distribution.png'

Performing advanced clustering analysis...
Vectorizing text data...
Applying K-means clustering...

Top keywords for each cluster:
Cluster 0 (13 answers): चक, गणन, गण, गए, खन, कह, कस, कव, सक, eot_id
Cluster 1 (8 answers): रचन, करन, रह, उपय, बल, रत, रक, इट, करत, मग
Cluster 2 (353 answers): करन, रत, सन, टन, अल, रक, इस, सफल, पहल, आव
Cluster 3 (14 answers): वसन, लर, करत, जन, गन, रम, अल, चयन, रत, करन
Cluster 4 (13 answers): गत, गण, गए, खन, कह, कस, कव, कल, करन, धर
Cluster 5 (14 answers): जर, गत, मन, यम, शन, छल, रह, अन, गल, षत
Cluster 6 (33 answers): मग, सम, टर, हम, नवर, रह, डल, रज, बड, सबस
Cluster 7 (17 answers): कल, करत, वर, पहल, रक, करन, डबल, रम, रचन, एनए
Cluster 8 (26 answers): इट, पпо, टल, इल, एल, वह, सफल, आव, पहल, टर
Cluster 9 (4 answers): गणन, गण, गए, खन, कह, कस, कव, कल, पहल, सफ

Generating cluster visualization...
Cluster visualization saved as 'answer_clusters.p

In [ ]:
import json


In [ ]:
cross_tab.to_csv("cluster_domain_crosstab.csv", encoding="utf-8-sig")
print("Cluster vs Domain cross-tabulation saved to 'cluster_domain_crosstab.csv'")

print("\nAnalysis complete!")

Cluster vs Domain cross-tabulation saved to 'cluster_domain_crosstab.csv'

Analysis complete!


In [ ]:
with open("cluster_domain_mapping.json", "w", encoding="utf-8") as f:
    json.dump(cluster_domain_mapping, f, ensure_ascii=False, indent=4)
print("Cluster-domain mapping saved to 'cluster_domain_mapping.json'")

Cluster-domain mapping saved to 'cluster_domain_mapping.json'


In [ ]:
# Add this import at the top of your script
import pandas as pd
import openpyxl
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment

# After your clustering code, add these functions:

def assign_cluster_names(df_subset, cluster_keywords):
    """
    Assigns meaningful names to clusters based on top keywords and domain distribution
    """
    # Create a mapping of cluster IDs to cluster names
    cluster_names = {}

    for cluster_id in range(len(cluster_keywords)):
        # Get the most common domain for this cluster
        cluster_domains = df_subset[df_subset["Cluster"] == cluster_id]["Domain"].value_counts()

        if not cluster_domains.empty:
            most_common_domain = cluster_domains.index[0]
            domain_count = cluster_domains.iloc[0]
            total_count = len(df_subset[df_subset["Cluster"] == cluster_id])
            domain_percentage = (domain_count / total_count) * 100

            # Get top keywords for this cluster
            top_words = cluster_keywords[str(cluster_id)] if str(cluster_id) in cluster_keywords else []

            # Create cluster name based on domain and top keywords
            if most_common_domain != "Other" and domain_percentage > 30:
                # If there's a clear domain, use it in the name along with top keywords
                cluster_names[cluster_id] = f"{most_common_domain}: {', '.join(top_words[:3])}"
            else:
                # If domain is "Other" or not clear, use only keywords
                cluster_names[cluster_id] = f"Cluster {cluster_id}: {', '.join(top_words[:5])}"

    # Apply cluster names to the dataframe
    df_subset["Cluster_Name"] = df_subset["Cluster"].map(cluster_names)

    return df_subset, cluster_names

def export_to_excel(df_subset, cluster_names, filename="qa_clusters.xlsx"):
    """
    Exports the data to Excel with formatted cluster names
    """
    # Create a writer object
    writer = pd.ExcelWriter(filename, engine='openpyxl')

    # Write the main data to the first sheet
    df_subset.to_excel(writer, sheet_name='Clustered Data', index=False)

    # Create a cluster summary sheet
    cluster_summary = []
    for cluster_id, name in cluster_names.items():
        cluster_data = df_subset[df_subset["Cluster"] == cluster_id]
        domains = cluster_data["Domain"].value_counts().to_dict()
        top_domain = max(domains.items(), key=lambda x: x[1])[0] if domains else "N/A"

        summary = {
            "Cluster ID": cluster_id,
            "Cluster Name": name,
            "Number of Answers": len(cluster_data),
            "Most Common Domain": top_domain,
            "Domain Count": domains.get(top_domain, 0),
            "Domain Percentage": round((domains.get(top_domain, 0) / len(cluster_data)) * 100, 2) if len(cluster_data) > 0 else 0
        }
        cluster_summary.append(summary)

    # Convert to dataframe and write to Excel
    summary_df = pd.DataFrame(cluster_summary)
    summary_df.to_excel(writer, sheet_name='Cluster Summary', index=False)

    # Format the Excel file
    workbook = writer.book

    # Format the cluster summary sheet
    summary_sheet = workbook['Cluster Summary']

    # Format header
    for cell in summary_sheet[1]:
        cell.font = Font(bold=True)
        cell.fill = PatternFill(start_color="D9D9D9", end_color="D9D9D9", fill_type="solid")
        cell.alignment = Alignment(horizontal='center', vertical='center')

    # Auto-adjust column widths
    for column in summary_sheet.columns:
        max_length = 0
        column_letter = openpyxl.utils.get_column_letter(column[0].column)
        for cell in column:
            try:
                if len(str(cell.value)) > max_length:
                    max_length = len(str(cell.value))
            except:
                pass
        adjusted_width = (max_length + 2) * 1.2
        summary_sheet.column_dimensions[column_letter].width = adjusted_width

    # Save the Excel file
    writer.close()
    print(f"Data exported to '{filename}' with formatted cluster names")

# Add after your clustering analysis:
# Convert integer cluster IDs to strings for JSON
cluster_keywords_str = {str(k): v for k, v in cluster_keywords.items()}

# Assign meaningful names to clusters
df_subset, cluster_names = assign_cluster_names(df_subset, cluster_keywords_str)

# Export to Excel
export_to_excel(df_subset, cluster_names, "hindi_qa_clusters.xlsx")

# Also save the cluster names to JSON
with open("cluster_names.json", "w", encoding="utf-8") as f:
    json.dump(cluster_names, f, ensure_ascii=False, indent=4)
print("Cluster names saved to 'cluster_names.json'")

# Add a function to analyze why many answers are classified as "Other"
def analyze_other_category(df):
    """
    Analyzes why many answers are being classified as "Other"
    """
    other_answers = df[df["Domain"] == "Other"]

    print(f"\nAnalyzing 'Other' category ({len(other_answers)} answers):")

    # Check length of answers
    answer_lengths = other_answers["Predicted_Answer"].str.len()
    print(f"Average length of 'Other' answers: {answer_lengths.mean():.1f} characters")
    print(f"Short answers (< 50 chars): {len(answer_lengths[answer_lengths < 50])} ({len(answer_lengths[answer_lengths < 50])/len(other_answers)*100:.1f}%)")

    # Sample some "Other" answers
    print("\nSample 'Other' answers:")
    for i, (_, row) in enumerate(other_answers.sample(min(5, len(other_answers))).iterrows()):
        print(f"Sample {i+1}: {row['Predicted_Answer'][:100]}...")

    # Suggestion for improvement
    print("\nPossible reasons for high 'Other' classification:")
    print("1. Domain keywords may not be comprehensive enough")
    print("2. Answers may contain mixed domains or specialized content")
    print("3. Very short or generic answers may lack domain-specific keywords")
    print("4. Non-textual or malformed answers might be present")

    # Save the analysis
    with open("other_category_analysis.txt", "w", encoding="utf-8") as f:
        f.write(f"Analysis of 'Other' category ({len(other_answers)} answers):\n")
        f.write(f"Average length of 'Other' answers: {answer_lengths.mean():.1f} characters\n")
        f.write(f"Short answers (< 50 chars): {len(answer_lengths[answer_lengths < 50])} ({len(answer_lengths[answer_lengths < 50])/len(other_answers)*100:.1f}%)\n\n")

        f.write("Sample 'Other' answers:\n")
        for i, (_, row) in enumerate(other_answers.sample(min(10, len(other_answers))).iterrows()):
            f.write(f"Sample {i+1}: {row['Predicted_Answer'][:200]}...\n\n")

        f.write("\nSuggestions to improve domain classification:\n")
        f.write("1. Expand the domain keywords list with more terms\n")
        f.write("2. Create new domains for common topics not currently covered\n")
        f.write("3. Implement a more sophisticated classification approach (e.g., embedding-based)\n")
        f.write("4. Create a pre-processing step to filter out very short or malformed answers\n")

    print("Analysis of 'Other' category saved to 'other_category_analysis.txt'")

# Call the analysis function at the end of your script
analyze_other_category(df)

Data exported to 'hindi_qa_clusters.xlsx' with formatted cluster names
Cluster names saved to 'cluster_names.json'

Analyzing 'Other' category (236 answers):
Average length of 'Other' answers: 63.3 characters
Short answers (< 50 chars): 88 (37.3%)

Sample 'Other' answers:
Sample 1: ब्लू वेल्फ को सबसे बड़ा...
Sample 2: एक ब्रिटिश चिकित्सक ने पेनिसिलिन की खोज की।<|eot_id...
Sample 3: पहला सफल पोलियो वैक्सीन किसने विकसित किया?<...
Sample 4: चार्ल्स एडेंसन ने पहले व्यावहारिक कैलकुलेटर का आविष्कार किया। \ n...
Sample 5: कौन सा पेड़ को पेड़ के सबसे बड़े पेड़ के रूप में से एक माना ज...

Possible reasons for high 'Other' classification:
1. Domain keywords may not be comprehensive enough
2. Answers may contain mixed domains or specialized content
3. Very short or generic answers may lack domain-specific keywords
4. Non-textual or malformed answers might be present
Analysis of 'Other' category saved to 'other_category_analysis.txt'


In [ ]:
# -*- coding: utf-8 -*-
"""
Answer Domain Classification Script (Fixed)
This script classifies Hindi QA answers into different knowledge domains
"""

import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from collections import Counter
from tqdm import tqdm

# Load the CSV file with predictions
print("Loading prediction data...")
df = pd.read_csv("predictions(pre-trained+finetuned).csv", encoding="utf-8-sig")

# Display sample data
print(f"Loaded {len(df)} predictions")
print("\nSample data:")
print(df[["Question", "Predicted_Answer"]].head(3))

# Define domain keywords (Hindi & English)
domain_keywords = {
    "History": ["इतिहास", "राजा", "महाराजा", "युद्ध", "साम्राज्य", "राज्य", "प्राचीन", "स्वतंत्रता",
                "emperor", "king", "dynasty", "war", "ancient", "freedom", "independence", "revolt", "kingdom",
                "शासक", "शासन", "राजवंश", "सम्राट", "क्रांति", "औपनिवेशिक", "आंदोलन", "सत्याग्रह", "संग्राम"],

    "Geography": ["भूगोल", "नदी", "पहाड़", "महासागर", "राज्य", "देश", "प्रदेश", "राजधानी",
                 "mountain", "river", "ocean", "country", "state", "capital", "continent", "climate", "plateau", "geography",
                 "भौगोलिक", "पर्वत", "समुद्र", "झील", "मैदान", "जलवायु", "मौसम", "तापमान", "वर्षा", "मरुस्थल"],

    "Politics": ["राजनीति", "प्रधानमंत्री", "मुख्यमंत्री", "राष्ट्रपति", "संसद", "लोकसभा", "राज्यसभा",
                "minister", "president", "parliament", "constitution", "government", "election", "party", "politics",
                "नेता", "विधानसभा", "मंत्री", "सरकार", "संविधान", "चुनाव", "राजनैतिक", "मतदान", "नीति"],

    "Biology": ["जीव विज्ञान", "जीव", "कोशिका", "डीएनए", "आनुवंशिकता", "डार्विन", "जीन", "कोशिका", "सेल",
               "biology", "cell", "DNA", "RNA", "genetics", "organism", "species", "evolution", "darwin", "chromosome",
               "प्रजाति", "जीवाश्म", "विकास", "जीवों", "मानव शरीर", "पारिस्थितिकी", "प्रोटीन", "वनस्पति"],

    "Physics": ["भौतिकी", "गति", "ऊर्जा", "प्रकाश", "ध्वनि", "न्यूटन", "आइंस्टाइन", "गुरुत्वाकर्षण",
               "physics", "motion", "energy", "light", "sound", "newton", "einstein", "gravity", "relativity",
               "आपेक्षिकता", "परमाणु", "विद्युत", "चुंबकत्व", "सापेक्षता", "क्वांटम", "प्रकाश", "ऊष्मा"],

    "Chemistry": ["रसायन", "तत्व", "यौगिक", "परमाणु", "अणु", "आवर्त सारणी", "अम्ल", "क्षार",
                 "chemistry", "element", "compound", "atom", "molecule", "periodic table", "acid", "base",
                 "H2O", "CO2", "रासायनिक", "धातु", "अधातु", "मिश्रण", "पदार्थ", "रासायनिक प्रतिक्रिया"],

    "Astronomy": ["खगोल", "सौर मंडल", "ग्रह", "सूरज", "चंद्रमा", "तारे", "आकाशगंगा", "ब्रह्मांड",
                 "astronomy", "solar system", "planet", "sun", "moon", "star", "galaxy", "universe",
                 "जुपिटर", "मंगल", "शनि", "बृहस्पति", "पृथ्वी", "नक्षत्र", "उल्का", "ग्रहण"],

    "Sports": ["खेल", "क्रिकेट", "फुटबॉल", "हॉकी", "खिलाड़ी",
              "cricket", "football", "hockey", "player", "tournament", "championship", "medal", "olympics", "sports", "game", "athlete",
              "प्रतियोगिता", "विश्व कप", "मैदान", "खेलकूद", "बल्लेबाज", "गेंदबाज", "टीम", "कप्तान"],

    "Economics": ["अर्थशास्त्र", "अर्थव्यवस्था", "मुद्रा", "बैंक", "वित्त",
                 "economy", "finance", "bank", "currency", "market", "trade", "economics", "budget", "gdp", "economic", "financial",
                 "बाजार", "व्यापार", "मूल्य", "निवेश", "मुद्रास्फीति", "आर्थिक", "पूंजी", "लाभ", "हानि"],

    "Culture": ["संस्कृति", "त्योहार", "परंपरा", "भाषा", "साहित्य", "कला",
               "culture", "festival", "tradition", "language", "literature", "art", "custom", "heritage", "folk", "dance", "music",
               "रीति-रिवाज", "लोकगीत", "नृत्य", "संगीत", "धरोहर", "परम्परा", "रस्म", "उत्सव"],

    "Religion": ["धर्म", "मंदिर", "मस्जिद", "चर्च", "पूजा",
                "religion", "temple", "mosque", "church", "worship", "god", "deity", "spiritual", "holy", "sacred", "prayer", "ritual",
                "भगवान", "देवता", "आस्था", "विश्वास", "पवित्र", "मोक्ष", "आत्मा", "धार्मिक", "प्रार्थना"],

    "Technology": ["प्रौद्योगिकी", "कंप्यूटर", "इंटरनेट", "सॉफ्टवेयर", "हार्डवेयर",
                  "computer", "internet", "software", "hardware", "technology", "digital", "electronic", "program", "device", "app",
                  "मोबाइल", "लैपटॉप", "नेटवर्क", "डिजिटल", "एप्लिकेशन", "डाटा", "प्रोग्रामिंग", "आर्टिफिशियल इंटेलिजेंस"],

    "Medicine": ["चिकित्सा", "डॉक्टर", "बीमारी", "रोग", "स्वास्थ्य", "अस्पताल", "दवा",
                "medicine", "doctor", "disease", "illness", "health", "hospital", "drug", "treatment", "cure",
                "रोगी", "इलाज", "उपचार", "शरीर", "अंग", "रक्त", "हृदय", "फेफड़े", "मस्तिष्क"],

    "Mathematics": ["गणित", "संख्या", "बीजगणित", "ज्यामिति", "कैलकुलस", "समीकरण",
                   "mathematics", "number", "algebra", "geometry", "calculus", "equation", "formula", "arithmetic",
                   "त्रिकोणमिति", "गणना", "सूत्र", "जोड़", "घटाव", "गुणा", "भाग", "आंकड़े", "संभावना"],
}

def classify_domain(text):
    """
    Classify a text into one of the predefined domains based on keyword presence
    Returns the domain with the highest keyword match count
    """
    # Convert to lowercase for easier matching
    if not isinstance(text, str):
        return "Unknown"

    text = text.lower()

    # Count matches for each domain
    domain_scores = {}
    for domain, keywords in domain_keywords.items():
        score = sum(1 for keyword in keywords if keyword.lower() in text.lower())
        domain_scores[domain] = score

    # Return the domain with the highest score
    if max(domain_scores.values()) > 0:
        return max(domain_scores.items(), key=lambda x: x[1])[0]
    else:
        return "Other"  # If no keywords match

# Apply classification to each predicted answer
print("\nClassifying answers by domain...")
tqdm.pandas(desc="Processing")
df["Domain"] = df["Predicted_Answer"].progress_apply(classify_domain)

# Count the number of answers in each domain
domain_counts = df["Domain"].value_counts()
print("\nDomain distribution:")
print(domain_counts)

# Save the classified data
df.to_csv("classified_predictions_llama3.csv", index=False, encoding="utf-8-sig")
print("\nClassified data saved to 'classified_predictions_llama3.csv'")

# Generate a bar chart of domain distribution
plt.figure(figsize=(12, 6))
domain_counts.plot(kind='bar', color='skyblue')
plt.title('Distribution of Answer Domains')
plt.xlabel('Domain')
plt.ylabel('Number of Answers')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('domain_distribution.png')
plt.close()

print("Domain distribution chart saved as 'domain_distribution.png'")

# Optional: Advanced clustering analysis
print("\nPerforming advanced clustering analysis...")

# Vectorize the text data
print("Vectorizing text data...")
# Only include answers that have at least 5 characters
valid_answers = df[df["Predicted_Answer"].str.len() > 5]["Predicted_Answer"].fillna("")
vectorizer = TfidfVectorizer(max_features=1000, stop_words=['और', 'है', 'हैं', 'का', 'के', 'की', 'में', 'से', 'पर', 'एक', 'यह', 'कि', 'थे', 'थी', 'थी', 'हो', 'था'])
X = vectorizer.fit_transform(valid_answers)

# Apply K-means clustering
print("Applying K-means clustering...")
n_clusters = min(10, len(valid_answers))  # Ensure we don't have more clusters than samples
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
df_subset = df[df["Predicted_Answer"].str.len() > 5].copy()
df_subset["Cluster"] = kmeans.fit_predict(X)

# Generate top words for each cluster
cluster_keywords = {}
feature_names = vectorizer.get_feature_names_out()

print("\nTop keywords for each cluster:")
for i in range(n_clusters):
    cluster_docs = df_subset[df_subset["Cluster"] == i]["Predicted_Answer"]
    if len(cluster_docs) > 0:
        cluster_tfidf = vectorizer.transform(cluster_docs)
        # Fix: Use numpy's mean function on the sparse matrix after converting to array
        tfidf_mean = np.mean(cluster_tfidf.toarray(), axis=0)

        # Get indices of top 10 words
        top_indices = np.argsort(tfidf_mean)[-10:]

        # Get the words corresponding to these indices
        top_words = [feature_names[idx] for idx in top_indices]

        cluster_keywords[i] = top_words
        print(f"Cluster {i} ({len(cluster_docs)} answers): {', '.join(top_words)}")

# Generate a scatter plot of the clusters
print("\nGenerating cluster visualization...")
pca = PCA(n_components=2)
X_dense = X.toarray()
pca_result = pca.fit_transform(X_dense)

plt.figure(figsize=(12, 8))
plt.scatter(pca_result[:, 0], pca_result[:, 1], c=df_subset["Cluster"], cmap='viridis', alpha=0.5)
plt.title('PCA Cluster Visualization of Answers')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.colorbar(label='Cluster')
plt.savefig('answer_clusters.png')
plt.close()

print("Cluster visualization saved as 'answer_clusters.png'")

# Optional: Compare manual classification with clustering
print("\nComparing manual classification with clustering...")
cluster_domain_mapping = {}
for cluster in range(n_clusters):
    cluster_domains = df_subset[df_subset["Cluster"] == cluster]["Domain"].value_counts()
    if not cluster_domains.empty:
        cluster_domain_mapping[cluster] = cluster_domains.index[0]
        print(f"Cluster {cluster} most common domain: {cluster_domains.index[0]} ({cluster_domains.iloc[0]} answers)")

# Create a cross-tabulation
cross_tab = pd.crosstab(df_subset["Cluster"], df_subset["Domain"])
print("\nCluster vs Domain Cross-tabulation:")
print(cross_tab)

print("\nAnalysis complete!")

Loading prediction data...
Loaded 500 predictions

Sample data:
                                     Question  \
0  पहले सफल ऑटोमोबाइल का आविष्कार किसने किया?   
1            डीएनए की संरचना की खोज किसने की?   
2             टेलीफोन का आविष्कार किसने किया?   

                                    Predicted_Answer  
0  किंग लेविस ने पहले सफल ऑटोमोबाइल का आविष्कार क...  
1           जेनेटिक पारमा डीएनए की संरचना की खोज की।  
2  अलेक्संडर ग्रेहम बेल टेलीफोन के आविष्कार के लि...  

Classifying answers by domain...


Processing: 100%|██████████| 500/500 [00:00<00:00, 1139.29it/s]



Domain distribution:
Domain
Other          236
Physics        107
Biology         70
Technology      22
Astronomy       18
Religion        14
Chemistry       12
Sports           5
Mathematics      4
Geography        4
Medicine         3
Unknown          2
Culture          1
History          1
Politics         1
Name: count, dtype: int64

Classified data saved to 'classified_predictions_llama3.csv'
Domain distribution chart saved as 'domain_distribution.png'

Performing advanced clustering analysis...
Vectorizing text data...
Applying K-means clustering...

Top keywords for each cluster:
Cluster 0 (13 answers): गण, गए, खन, कह, कस, कव, करन, हस, सक, eot_id
Cluster 1 (8 answers): रचन, करन, रह, उपय, बल, रत, रक, इट, करत, मग
Cluster 2 (353 answers): करन, रत, सन, टन, अल, रक, इस, सफल, पहल, आव
Cluster 3 (14 answers): वसन, लर, करत, जन, गन, रम, अल, चयन, रत, करन
Cluster 4 (13 answers): हस, गए, खन, कह, कस, कव, कल, करत, करन, धर
Cluster 5 (14 answers): जर, गत, मन, यम, शन, छल, रह, अन, गल, षत
Cluster 6